# H4 — Acquisition‑Order Alignment Analysis

End‑to‑end notebook for the H4 hypothesis test (`docs/new_rules/improved_research_context_v5.md §3 / §8 / §23.12`).

**Inputs (in Google Drive after `notebooks/evaluation.ipynb` Cell 7 finished):**
- `MyDrive/Researchs/BabyLM/submissions/babylm-2026-taam_v2-seed42_predictions.json`
- `MyDrive/Researchs/BabyLM/submissions/babylm-2026-b0-seed42_predictions.json`

Both files must contain a non‑empty `fast_eval_results` field (24 entries).

**Outputs (written back to the repo + Drive):**
- `analyses/results/rho_table.csv` — Spearman ρ per (condition, language)
- `analyses/results/delta_rho_table.csv` — paired bootstrap CI on `ρ_TAAM-v2 − ρ_B0`
- `analyses/results/zh_scatter.csv` — per‑phenomenon ZH scatter rows
- `analyses/results/t_acq_table.csv` — every (cond, seed, lang, phen, t_acq, child_age) row
- `runs/_h4/audit.csv` — per‑task accuracy averaging audit
- `analyses/figures/figure6_acquisition_order.png` — the canonical H4 figure

**One‑line H4 claim:** *TAAM‑v2's acquisition order on per‑phenomenon minimal pairs correlates more strongly with child age‑of‑acquisition norms than B0's, especially on Chinese.*

**Statistic:** `ρ_TAAM‑v2 − ρ_B0` with paired bootstrap 95 % CI, Holm‑Bonferroni adjusted over the 3 languages.

## Cell 1 — Parameters (edit before running)

In [ ]:
# ============================================================
# Where the lm-eval RAW results live (PREFERRED — full sub-task granularity)
# ============================================================
# The H4 analysis needs per-phenomenon accuracies, so it reads the raw
# `eval_results/chck_NM/<slug>/results_*.json` files that Cell 6 of
# `notebooks/evaluation.ipynb` writes (those have all ~110 sub-tasks).
# The aggregated `*_predictions.json` files (Cell 7) only have ~16 leaderboard
# tasks and are NOT enough for H4 (this was the EmptyDataError / N=0 cause).
USE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/Researchs/BabyLM"

# === preferred: raw lm-eval results dir on Drive ===
EVAL_RESULTS_DIR = f"{DRIVE_ROOT}/eval_results"   # contains chck_*M/.../results_*.json
H4_MODEL_IDS = [
    "taam_v2-seed42",   # condition A (typology-aware)
    "b0-seed42",        # condition B (uniform baseline)
]

# === fallback: aggregated predictions JSON (uses LESS info — last resort) ===
PREDICTIONS_DIR = f"{DRIVE_ROOT}/submissions"
H4_MODELS_FALLBACK = [
    f"babylm-2026-{m}" for m in H4_MODEL_IDS
]

# Cell 4 will auto-detect which source is available and set this flag:
SOURCE_MODE = None  # "eval-results-dir" | "predictions-glob"

# Pipeline knobs
# t_acq method: 'relative' = Evanson et al. 2023 (step to reach 90% of the
#   model's OWN final accuracy on a probe; only above-chance probes count).
#   This is the anchor-paper-faithful method and the reason N is no longer <6.
# 'absolute' = legacy fixed threshold (kept only for the appendix robustness row).
T_ACQ_METHOD    = "relative"  # "relative" (default, Evanson) | "absolute" (legacy)
RELATIVE_FRAC   = 0.90   # for relative method: fraction of final accuracy
THRESHOLD       = 0.70   # for absolute method ONLY
CONSECUTIVE     = 2      # how many consecutive evals must pass the criterion
N_MIN           = 6      # minimum N (acquired phenomena per language) to compute ρ
N_BOOTSTRAP     = 10_000 # paired-bootstrap iterations
# Child norms + task map: single CSV source of truth (docs/h4_child_norms.md).
NORMS_CSV       = "analyses/acquisition_order/child_norms_dataset.csv"

# Repo layout (script paths are relative to REPO_ROOT once we cd into the repo)
REPO_URL    = "https://github.com/Amos-Luna/Asymmetric-Multilingual-Acquisition_TAAM.git"   # only used if not already cloned
REPO_BRANCH = "main"

# ============================================================
# Output paths
# ============================================================
# IN-REPO (volatile in Colab — survives only this session):
OUT_RUNS_DIR    = "runs/_h4"
OUT_RESULTS_DIR = "analyses/results"
OUT_FIGURES_DIR = "analyses/figures"

# IN-DRIVE (persistent across Colab disconnects — primary storage for outputs):
PERSIST_TO_DRIVE = True
DRIVE_OUT_RUNS   = f"{DRIVE_ROOT}/h4_outputs/runs"      # eval_results.json + audit.csv
DRIVE_OUT_TABLES = f"{DRIVE_ROOT}/h4_outputs/tables"    # rho_table.csv, delta_rho_table.csv, ...
DRIVE_OUT_FIGS   = f"{DRIVE_ROOT}/h4_outputs/figures"   # figure6_*.png

print("H4 plan:")
print(f"  eval-results dir   : {EVAL_RESULTS_DIR}")
print(f"  H4 models          : {H4_MODEL_IDS}")
print(f"  t_acq method       : {T_ACQ_METHOD} (frac={RELATIVE_FRAC})")
print(f"  norms CSV          : {NORMS_CSV}")
print(f"  bootstrap iters    : {N_BOOTSTRAP}")
print(f"  in-repo tables     : {OUT_RESULTS_DIR}")
print(f"  Drive persistence  : {PERSIST_TO_DRIVE}")
if PERSIST_TO_DRIVE:
    print(f"    runs    -> {DRIVE_OUT_RUNS}")
    print(f"    tables  -> {DRIVE_OUT_TABLES}")
    print(f"    figures -> {DRIVE_OUT_FIGS}")

## Cell 2 — Environment setup

Installs `pyyaml`, `numpy`, `scipy`, `matplotlib` if not present. Clones the repo if we're not already inside it.

In [ ]:
import os, subprocess, sys, importlib
from pathlib import Path

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *pkgs])

for pkg in ("pyyaml", "numpy", "scipy", "matplotlib", "pandas"):
    try:
        __import__("yaml" if pkg == "pyyaml" else pkg)
    except ImportError:
        print(f"installing {pkg}...")
        _pip(pkg)

# Locate the repo. If we are running in Colab and not yet in the repo, clone it.
REPO_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "scripts" / "h4" / "run_h4_pipeline.py").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    REPO_ROOT = Path("/content/Asymmetric-Multilingual-Acquisition_TAAM")
    if not REPO_ROOT.exists():
        print(f"Cloning repo to {REPO_ROOT}...")
        subprocess.check_call([
            "git", "clone", "--depth=1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT),
        ])

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

# ── ALWAYS refresh the repo (so re-running this notebook picks up fixes) ────
# This is what fixes the "got an unexpected keyword argument 'eval_results_dir'"
# class of errors: the Colab disk has a stale clone of the repo.
try:
    subprocess.check_output(["git", "-C", str(REPO_ROOT), "fetch", "--quiet"],
                            stderr=subprocess.STDOUT)
    subprocess.check_output(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "--quiet"],
                            stderr=subprocess.STDOUT)
    print("git pull --ff-only: OK")
except subprocess.CalledProcessError as e:
    out = e.output.decode("utf-8", errors="replace") if e.output else ""
    if "diverged" in out or "non-fast-forward" in out or "would be overwritten" in out:
        print(f"⚠️  git pull failed (local divergence). Output:\n{out}")
        print("   Skipping pull — using whatever is on disk.")
    else:
        # Repo may be a non-git directory (e.g. ZIP download). Not fatal.
        print(f"(could not git-pull: {out.strip()[:200]} — using disk version)")

# Show the SHA actually in use so you can spot stale clones at a glance.
try:
    sha = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "log", "-1", "--format=%h  %ad  %s",
         "--date=short"],
        stderr=subprocess.DEVNULL,
    ).decode("utf-8").strip()
    print(f"repo HEAD: {sha}")
except subprocess.CalledProcessError:
    pass

# ── Drop any cached `scripts.*` modules so re-importing picks up new code ───
# Without this, re-running Cell 5 after a git pull would still use the old
# in-memory `run_pipeline` (the function signature would not match).
for mod in list(sys.modules):
    if mod == "scripts" or mod.startswith("scripts."):
        del sys.modules[mod]
importlib.invalidate_caches()

print(f"REPO_ROOT = {REPO_ROOT}")

# Sanity: confirm the YAMLs and (critical!) the new run_pipeline signature.
for p in ("analyses/acquisition_order/phenomenon_to_child_norm.yaml",
         "analyses/acquisition_order/phenomenon_to_task_map.yaml",
         "scripts/h4/run_h4_pipeline.py",
         "scripts/analyze_acquisition_order.py"):
    exists = (REPO_ROOT / p).exists()
    print(f"  {'✓' if exists else '✗'} {p}")

# Verify the loaded `run_pipeline` accepts `eval_results_dir` (the bug fix).
import inspect
from scripts.h4.run_h4_pipeline import run_pipeline as _rp
params = list(inspect.signature(_rp).parameters)
ok = "eval_results_dir" in params and "h4_models" in params and "mirror_dirs" in params
print(f"  {'✓' if ok else '✗'} run_pipeline signature accepts: "
      f"eval_results_dir={('eval_results_dir' in params)}, "
      f"h4_models={('h4_models' in params)}, "
      f"mirror_dirs={('mirror_dirs' in params)}")
if not ok:
    print("    → STOP. Your repo clone is stale. Open a fresh runtime "
          "(Runtime → Disconnect and delete runtime) and re-run all cells.")
    print(f"    actual params: {params}")

## Cell 3 — Mount Google Drive (Colab only)

In [ ]:
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        print(f"Drive mounted. Looking for predictions in {PREDICTIONS_DIR}")
    except ImportError:
        print("Not in Colab; assuming Drive is already mounted or paths are local.")
else:
    print("USE_DRIVE = False — assuming predictions JSONs are already in the repo.")

## Cell 4 — Locate inputs (auto-detect raw eval dir vs. predictions JSON)
This cell picks the **best available source**: it prefers the raw
`eval_results/chck_*M/<slug>/results_*.json` files (full sub-task granularity,
which is what H4 needs); if those are missing it falls back to the aggregated
`*_predictions.json` files (low fidelity — will yield N=0 for most phenomena).

Confirms both H4 models have a non‑empty `fast_eval_results` block. If not, **stop here** and re‑run `evaluation.ipynb` Cell 6 (Fast Eval) on the missing model.

In [ ]:
import json, re
from pathlib import Path

eval_dir = Path(EVAL_RESULTS_DIR)
pred_paths = []
SOURCE_MODE = None
revisions_on_disk = []

# ── Try mode 1 first: raw eval_results/chck_*M/<slug>/results_*.json ────────
if eval_dir.exists():
    # Auto-discover revision dirs (chck_NM, skipping `main` which has no
    # numeric token count and cannot be ordered for Spearman).
    all_subs = sorted(p for p in eval_dir.iterdir() if p.is_dir())
    revisions_on_disk = [p.name for p in all_subs if re.match(r"^chck_\d+M$", p.name)]
    revisions_on_disk.sort(key=lambda r: int(re.search(r"\d+", r).group()))
    other = [p.name for p in all_subs if p.name not in revisions_on_disk]
    print(f"Eval-results dir: {eval_dir}")
    print(f"  numeric checkpoints found: {len(revisions_on_disk)}")
    if revisions_on_disk:
        print(f"    {revisions_on_disk}")
    if other:
        print(f"  ignored (non-numeric): {other}")

if revisions_on_disk:
    # For each model count: (a) how many revisions have a matching slug folder,
    # (b) how many of those have at least one results_*.json
    print()
    print("Per-model coverage of raw lm-eval results:")
    print(f"  {'model':25s}  {'rev_dirs':>8s}  {'with_json':>10s}  {'sample_file'}")
    coverage = {m: {"rev_dirs": 0, "with_json": 0, "sample": None} for m in H4_MODEL_IDS}
    for rev in revisions_on_disk:
        rev_dir = eval_dir / rev
        for m in H4_MODEL_IDS:
            # The slug we expect (double-underscore between owner and repo).
            candidates = [
                rev_dir / f"amosluna__babylm-2026-{m}",
                rev_dir / f"amosluna/babylm-2026-{m}",
                rev_dir / f"babylm-2026-{m}",
            ]
            # If exact paths don't match, fall back to a glob with substring.
            found_dirs = [c for c in candidates if c.exists()]
            if not found_dirs:
                found_dirs = list(rev_dir.glob(f"*{m}*"))
            if not found_dirs:
                continue
            coverage[m]["rev_dirs"] += 1
            for d in found_dirs:
                jsons = list(d.glob("results_*.json"))
                if jsons:
                    coverage[m]["with_json"] += 1
                    if coverage[m]["sample"] is None:
                        coverage[m]["sample"] = jsons[0]
                    break
    for m in H4_MODEL_IDS:
        c = coverage[m]
        flag = "✓" if c["with_json"] >= 6 else ("~" if c["with_json"] > 0 else "✗")
        sample = c["sample"].relative_to(eval_dir) if c["sample"] else "(none)"
        print(f"  {flag} {m:23s}  {c['rev_dirs']:>8d}  {c['with_json']:>10d}  {sample}")
    if all(coverage[m]["with_json"] > 0 for m in H4_MODEL_IDS):
        SOURCE_MODE = "eval-results-dir"
        print(f"\n→ Using SOURCE_MODE = '{SOURCE_MODE}' (preferred, full sub-task granularity).")

# ── Fallback to mode 2 if raw dir absent or empty ──────────────────────────
if SOURCE_MODE is None:
    print(f"\nRaw eval_results dir not usable. Falling back to *_predictions.json.")
    print("(Warning: aggregated tasks only — H4 will yield n=0 for almost everything.)")
    for model in H4_MODELS_FALLBACK:
        p = Path(PREDICTIONS_DIR) / f"{model}_predictions.json"
        if not p.exists():
            print(f"  ✗ MISSING: {p}")
            continue
        data = json.loads(p.read_text())
        fast = data.get("fast_eval_results") or []
        zs   = data.get("zeroshot") or {}
        flag = "✓" if fast else "✗"
        print(f"  {flag} {p.name}: {len(zs)} zeroshot tasks, {len(fast)} fast_eval revisions")
        if fast:
            pred_paths.append(p)
    if pred_paths:
        SOURCE_MODE = "predictions-glob"

print()
print(f"SOURCE_MODE = {SOURCE_MODE!r}")
if SOURCE_MODE is None:
    print("ERROR: no usable input source found. Verify EVAL_RESULTS_DIR or PREDICTIONS_DIR.")
else:
    print("Ready for Cell 5.")

## Cell 5 — Run the full H4 pipeline

Calls `scripts.h4.run_h4_pipeline.run_pipeline(...)`. Total wall‑clock is a few seconds (the bottleneck is the 10 000‑iteration paired bootstrap).

In [ ]:
from scripts.h4.run_h4_pipeline import run_pipeline

# Build the list of mirror directories so the pipeline copies every output
# to Drive on the way out (CSVs, Figure 6, and per-run eval_results.json).
# This is critical because Colab's /content disk is wiped between sessions.
mirror_dirs = []
if PERSIST_TO_DRIVE:
    Path(DRIVE_OUT_TABLES).mkdir(parents=True, exist_ok=True)
    mirror_dirs.append(Path(DRIVE_OUT_TABLES))

# v2: norms_csv is the single source of truth for BOTH task map and child
# ages. We do NOT pass task_map_path (that would force the legacy YAML).
common = dict(
    out_runs_dir=Path(OUT_RUNS_DIR),
    out_results_dir=Path(OUT_RESULTS_DIR),
    out_figures_dir=Path(OUT_FIGURES_DIR),
    norms_csv=Path(NORMS_CSV),
    t_acq_method=T_ACQ_METHOD,
    relative_frac=RELATIVE_FRAC,
    threshold=THRESHOLD,
    consecutive=CONSECUTIVE,
    n_min=N_MIN,
    n_bootstrap=N_BOOTSTRAP,
    cond_a="taam_v2",
    cond_b="b0",
    mirror_dirs=mirror_dirs,
)

if SOURCE_MODE == "eval-results-dir":
    summary = run_pipeline(
        eval_results_dir=Path(EVAL_RESULTS_DIR),
        h4_models=H4_MODEL_IDS,
        **common,
    )
elif SOURCE_MODE == "predictions-glob":
    summary = run_pipeline(
        predictions_paths=pred_paths,
        **common,
    )
else:
    raise RuntimeError("SOURCE_MODE not set — Cell 4 found no usable input source.")

print()
print(f"H4 pipeline completed (mode = {SOURCE_MODE}).")
print(f"  records collected     : {summary['n_records']}")
print(f"  rho rows (cond×lang)  : {len(summary['rho_rows'])}")
print(f"  delta_rho rows (lang) : {len(summary['delta_rows'])}")
print(f"  mirrored to           : {summary['mirrored_to']}")
for k, v in summary["outputs"].items():
    print(f"  {k:20s}: {v}")

# Sanity check — open audit.csv to see whether any phenomena reached threshold.
import pandas as pd
audit = pd.read_csv(Path(OUT_RUNS_DIR) / "audit.csv")
n_with_score = (audit["accuracy"] != "").sum() if "accuracy" in audit.columns else 0
n_total = len(audit)
print(f"\nAudit: {n_with_score}/{n_total} phen×lang×step cells have a numeric accuracy.")
if n_with_score == 0:
    print("⚠️  All cells empty — the task map does not match any task in your input.")
    print("   Open audit.csv and verify that 'n_tasks_valid' > 0 for at least some rows.")
elif summary["n_records"] == 0:
    print("⚠️  audit has data but no phenomena were acquired (n_records=0).")
    print(f"   Try lowering THRESHOLD (current = {THRESHOLD}) or CONSECUTIVE (current = {CONSECUTIVE}).")

## Cell 6 — Display the results

Renders the ρ table, the Δρ table, and Figure 6 inline. **Look at the `delta_rho_table.csv` row for `zho`** — that is the line that decides H4.

In [ ]:
import math
import pandas as pd
from IPython.display import Image, display

results_dir = Path(OUT_RESULTS_DIR)
figures_dir = Path(OUT_FIGURES_DIR)


def _safe_read_csv(path: Path) -> pd.DataFrame:
    """Read a CSV that might have only a header (returns empty DataFrame)."""
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()
    except FileNotFoundError:
        print(f"  (missing file: {path})")
        return pd.DataFrame()


# ──────────────────────────────────────────────────────────────────────────────
# ρ table — per (condition × language) correlation with child norms
# ──────────────────────────────────────────────────────────────────────────────
print("\n=== ρ table: Spearman correlation with child age-of-acquisition ===")
rho = _safe_read_csv(results_dir / "rho_table.csv")
display(rho)

# ──────────────────────────────────────────────────────────────────────────────
# Δρ table — TAAM-v2 minus B0, the headline number for H4
# ──────────────────────────────────────────────────────────────────────────────
print("\n=== Δρ table: TAAM-v2 minus B0, Holm-Bonferroni adjusted ===")
delta = _safe_read_csv(results_dir / "delta_rho_table.csv")
if delta.empty:
    print("  (no rows — likely because no phenomenon reached the acquisition threshold)")
else:
    display(delta)

# Headline H4 verdict
print("\n=== H4 verdict per language ===")
if delta.empty:
    print("  No verdict possible: the Δρ table is empty. Diagnostic checklist:")
    print(f"    1. audit.csv : open {Path(OUT_RUNS_DIR)/'audit.csv'} and check that")
    print("       'n_tasks_valid' > 0 for many rows. If all are 0, the task map")
    print("       does not match your lm-eval task names.")
    print("    2. THRESHOLD : current = {:.2f}. Lower it (e.g. to 0.60) in Cell 1".format(THRESHOLD))
    print("       and re-run if your models do not cross 70% sustained.")
    print("    3. N_MIN     : current = {}. With one seed per condition you may".format(N_MIN))
    print("       need N_MIN=4 to get a verdict on small languages.")
else:
    for _, row in delta.iterrows():
        lang = row["language"]
        try:
            d = float(row["delta"])
        except (TypeError, ValueError):
            d = float("nan")
        p_h = float(row["p_holm"]) if str(row["p_holm"]) not in ("", "nan") else float("nan")
        if math.isnan(p_h):
            verdict = "N too small to test"
        elif p_h < 0.05 and d > 0:
            verdict = f"H4 SUPPORTED on {lang}  (Δρ={d:+.3f}, p_holm={p_h:.3f})"
        elif p_h < 0.05 and d < 0:
            verdict = f"H4 REJECTED (wrong sign) on {lang}  (Δρ={d:+.3f}, p_holm={p_h:.3f})"
        else:
            verdict = f"H4 NULL on {lang}  (Δρ={d:+.3f}, p_holm={p_h:.3f})"
        print(f"  {lang}: {verdict}")

# ──────────────────────────────────────────────────────────────────────────────
# Audit snapshot — useful when ρ is NaN to find the root cause
# ──────────────────────────────────────────────────────────────────────────────
audit_path = Path(OUT_RUNS_DIR) / "audit.csv"
if audit_path.exists():
    audit = _safe_read_csv(audit_path)
    if not audit.empty:
        # numeric coercion for plotting / aggregation
        audit["accuracy_num"] = pd.to_numeric(audit["accuracy"], errors="coerce")
        print("\n=== Audit snapshot: max accuracy per (condition, language, phenomenon) ===")
        peak = (audit
                .dropna(subset=["accuracy_num"])
                .groupby(["condition", "language", "phenomenon_id"], as_index=False)
                ["accuracy_num"].max()
                .rename(columns={"accuracy_num": "max_accuracy"}))
        peak["acquired_at_70"] = peak["max_accuracy"] >= 0.70
        peak["acquired_at_60"] = peak["max_accuracy"] >= 0.60
        peak["acquired_at_50"] = peak["max_accuracy"] >= 0.50
        display(peak.sort_values(["condition", "language", "max_accuracy"],
                                  ascending=[True, True, False]))

# ──────────────────────────────────────────────────────────────────────────────
# Figure 6
# ──────────────────────────────────────────────────────────────────────────────
fig6_path = figures_dir / "figure6_acquisition_order.png"
print(f"\n=== Figure 6 ({fig6_path}) ===")
if fig6_path.exists():
    display(Image(str(fig6_path), width=900))
else:
    print("  (Figure 6 not found — check Cell 5 for errors)")

## Cell 7 — Persistence to Drive (redundant safety + per-run JSON copy)

Cell 5 already mirrors every CSV + Figure 6 + per-run `eval_results.json` to
`DRIVE_OUT_TABLES` while the pipeline runs. This cell performs a final
explicit copy into the three Drive output trees (`tables/`, `figures/`,
`runs/<cond>_seedN/`) so the outputs survive Colab disconnects, and prints
the final paths you can cite in the paper.

Copies the result CSVs + Figure 6 to `Drive/MyDrive/Researchs/BabyLM/h4_results/` so they survive Colab disconnects and are easy to share.

In [ ]:
import shutil

if not PERSIST_TO_DRIVE:
    print("PERSIST_TO_DRIVE = False — skipping Drive persistence.")
else:
    tables_dst  = Path(DRIVE_OUT_TABLES)
    figures_dst = Path(DRIVE_OUT_FIGS)
    runs_dst    = Path(DRIVE_OUT_RUNS)
    for d in (tables_dst, figures_dst, runs_dst):
        d.mkdir(parents=True, exist_ok=True)

    print(f"Persisting outputs to Drive:")
    print(f"  tables  -> {tables_dst}")
    print(f"  figures -> {figures_dst}")
    print(f"  runs    -> {runs_dst}")
    print()

    # 1) Tables (CSVs) — both pipeline-mirror & this explicit copy go to Drive
    for fname in ("rho_table.csv", "delta_rho_table.csv",
                  "zh_scatter.csv", "t_acq_table.csv"):
        src = Path(OUT_RESULTS_DIR) / fname
        if src.exists():
            shutil.copy2(src, tables_dst / fname)
            print(f"  ✓ {fname:25s} -> {tables_dst/fname}")
        else:
            print(f"  ✗ {fname:25s} not produced by Cell 5")

    # 2) Figure 6
    fig_src = Path(OUT_FIGURES_DIR) / "figure6_acquisition_order.png"
    if fig_src.exists():
        shutil.copy2(fig_src, figures_dst / fig_src.name)
        print(f"  ✓ {fig_src.name:25s} -> {figures_dst/fig_src.name}")

    # 3) Audit CSV (goes into tables/ since it is paper-appendix material)
    audit = Path(OUT_RUNS_DIR) / "audit.csv"
    if audit.exists():
        shutil.copy2(audit, tables_dst / "audit.csv")
        print(f"  ✓ {'audit.csv':25s} -> {tables_dst/'audit.csv'}")

    # 4) Per-run eval_results.json (so we can reload without re-running the bridge)
    runs_root = Path(OUT_RUNS_DIR)
    for run_dir in runs_root.glob("*_seed*"):
        src_json = run_dir / "eval_results.json"
        if src_json.exists():
            dst_sub = runs_dst / run_dir.name
            dst_sub.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src_json, dst_sub / src_json.name)
            print(f"  ✓ runs/{run_dir.name}/eval_results.json -> {dst_sub/src_json.name}")

    print(f"\nAll outputs persisted to {Path(DRIVE_ROOT) / 'h4_outputs'}.")
    print("They will survive Colab disconnects. To re-render figures without")
    print("re-running the bridge, point `OUT_RUNS_DIR` at the Drive runs/ folder.")

## What to do with these results

### If `delta_rho_table.csv` shows TAAM‑v2 > B0 on **zho** with `p_holm < 0.05`
→ **H4 supported on Chinese.** This is the headline. Add it to §5 of the paper, cite the row from `delta_rho_table.csv`, embed Figure 6.

### If TAAM‑v2 > B0 on **all three** languages
→ **Strongest possible H4 outcome.** Lead with this in §1 of the abstract. "TAAM‑v2 acquires grammar in an order significantly closer to child norms than the uniform baseline across three typologically distant languages."

### If signs disagree across languages (e.g. TAAM > B0 on zho, B0 > TAAM on eng)
→ **Publishable: a per‑language analysis of when adaptive mixing changes the acquisition trajectory.** Argue that ZH (the language that received the most boost from TAAM) is where the trajectory diverges most from B0, and that EN/NL stay near B0 because they were already on the dominant‑language trajectory. Cite Hoff 2012 and MacWhinney 1987.

### If all three languages are `p_holm > 0.05` or NaN
→ **Null H4: publishable as such.** Lift §3.3 of `improved_research_context_v5.md`: *"Online adaptive multilingual mixing changes what the model masters by 100 M tokens, but not the order in which it does so."* This is a legitimate scientific contribution — most BabyLM papers don't even *try* to test acquisition order.

### If `n_pairs` per language is small (< 8)
→ **Statistical power is the bottleneck.** Options:
1. Lower the threshold (`THRESHOLD = 0.65`) to acquire more phenomena.
2. Add more phenomena to `phenomenon_to_child_norm.yaml` and `phenomenon_to_task_map.yaml`.
3. Add a second seed of each H4 model and re‑run (would double the records but also double the compute on the trajectory side).

### If `n_pairs = 0` for every language (the `EmptyDataError` case)
→ **Look at the *audit snapshot* table that Cell 6 now prints.** It shows, per
condition × language × phenomenon, the maximum accuracy ever reached over the
24 fast‑eval checkpoints, plus three flags (`acquired_at_70/60/50`).

1. **All `max_accuracy` are near `NaN` or 0.5.** Your `phenomenon_to_task_map.yaml`
   probably does not match any task name in the actual lm‑eval results. Open
   `runs/_h4/audit.csv` and check `n_tasks_valid`; if it is 0 everywhere, fix
   the YAML.
2. **Many `max_accuracy ≥ 0.60` but few `≥ 0.70`.** Lower `THRESHOLD` in Cell 1
   to `0.65` or `0.60` and re‑run from Cell 5. Update the methods section of
   the paper to report the threshold used.
3. **`acquired_at_70 = True` for many rows, but Δρ is still `n_pairs = 0`.**
   The acquired phenomena differ between conditions — Δρ needs the *same*
   phenomenon acquired by *both* conditions. Lower the threshold so more
   phenomena are shared.
4. **`SOURCE_MODE = 'predictions-glob'` in Cell 4.** You are on the fallback
   mode that only has 16 aggregated tasks, not the 110 sub‑tasks H4 needs. Run
   the eval pipeline so `eval_results/chck_*M/<slug>/results_*.json` files
   exist on Drive, then re‑run Cell 4.

All decisions go in `docs/new_rules/improved_research_context_v5.md §23.12` (H4 status) and `docs/h4_papers.md §6` (honest assessment).

### Where do the outputs live?

When `PERSIST_TO_DRIVE = True` (default), Cell 5 mirrors **and** Cell 7
re‑mirrors all outputs to Drive at:

```
$DRIVE_ROOT/h4_outputs/
├── tables/
│   ├── rho_table.csv          ← per (condition, lang) Spearman ρ + CI + N + note
│   ├── delta_rho_table.csv    ← TAAM‑v2 minus B0 with Holm‑adjusted p
│   ├── zh_scatter.csv         ← per‑phenomenon (t_acq_step, child_age) for ZH
│   ├── t_acq_table.csv        ← raw t_acq per phenomenon × cond × seed × lang
│   └── audit.csv              ← per phen × lang × step accuracy trace
├── figures/
│   └── figure6_acquisition_order.png
└── runs/
    ├── taam_v2_seed42/eval_results.json
    └── b0_seed42/eval_results.json
```

These files survive Colab disconnects. To re‑render figures or re‑compute Δρ
without re‑running the bridge, point a new notebook session at
`DRIVE_ROOT/h4_outputs/runs/` and call `run_pipeline(...)` with the existing
`eval_results.json` files in place (the bridge will skip work it already did).